# 🛰️ Sentinel-2 Land Cover Classification Pipeline
### Kachin State Rare Earth Mine Detection

---

**Pipeline Overview:**

| Step | Description |
|------|-------------|
| 0 | Configuration & Imports |
| 1 | Load & Preprocess Raw GeoTIFF (Reproject → Scale → Resample) |
| 2 | Compute Spectral Indices |
| 3 | Feature Selection & Standardisation |
| 4 | K-Means Unsupervised Clustering |
| 5 | 4-Class Mine RF (with Shapefile Polygon) |
| 7 | Multi-Year Change Analysis |

**Input:** Raw 5-band Sentinel-2 GeoTIFF exported from GEE (B2, B3, B4, B8, B11 — DN values)  
**Output:** Classified land cover maps, area statistics, deforestation & mine growth trends

---
## Step 0 — Configuration & Imports

In [92]:
import numpy as np
import rasterio
from rasterio.crs import CRS
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.features import rasterize
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import pandas as pd
import geopandas as gpd
from shapely.geometry import mapping
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
import warnings, pickle, os
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

print('✅ All imports successful')

✅ All imports successful


In [93]:
# ── FILE PATHS ─────────────────────────────────────────────────────────────
INPUT_TIF       = '../region_of_interest/Sentinel2_2024_Kachin.tif'  # raw GEE export
MINE_SHP        = '../region_of_interest/mine_polygons.shp'           # mine boundary shapefile
OUTPUT_DIR      = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Multi-year TIFs — add/remove years as needed
YEARLY_TIFS = {
    2020: '../region_of_interest/Sentinel2_2020_Kachin.tif',
    2021: '../region_of_interest/Sentinel2_2021_Kachin.tif',
    2022: '../region_of_interest/Sentinel2_2022_Kachin.tif',
    2023: '../region_of_interest/Sentinel2_2023_Kachin.tif',
    2024: '../region_of_interest/Sentinel2_2024_Kachin.tif',
    2025: '../region_of_interest/Sentinel2_2025_Kachin.tif',
}
BASELINE_YEAR = 2020   # for change analysis

# ── PREPROCESSING ──────────────────────────────────────────────────────────
BAND_NAMES   = ['B2_Blue', 'B3_Green', 'B4_Red', 'B8_NIR', 'B11_SWIR_10m']  # band order in TIF
SCALE_FACTOR = 10_000   # GEE S2_SR_HARMONIZED DN → reflectance
TARGET_CRS   = CRS.from_epsg(32647)   # UTM 47N (Kachin State)
TARGET_RES   = 10.0                    # metres — all bands resampled to 10 m grid
SWIR_BANDS   = {'B11_SWIR_10m', 'B11_SWIR', 'B11'}  # bilinear-resampled (native 20 m)

# ── CLASSIFICATION ─────────────────────────────────────────────────────────
SELECTED_FEATURES = ['B8_NIR', 'B11_SWIR_10m', 'NDVI', 'EVI', 'SAVI', 'BSI', 'FCI']
N_CLUSTERS        = 4      # K-Means k  (merge duplicates into final 3 classes)
N_SAMPLES_PER_CLS = 5_000  # training samples per class for RF
N_ESTIMATORS      = 200    # RF trees
RANDOM_STATE      = 42

# ── MINE MASK THRESHOLDS (matching notebook) ───────────────────────────────
FOREST_NDVI_MIN  = 0.70   # forest training pixels: NDVI >= this
NDVI_CLEANUP     = 0.75   # post-classification cleanup: NDVI >= this → Forest
MINE_NDVI_MAX    = 0.55   # mine pixels: NDVI < this (exclude vegetated edges)
MINE_NDWI_MIN    = -0.22  # mine pixels: NDWI > this (retain leachate ponds)

# ── COLOURS ────────────────────────────────────────────────────────────────
CLASS_COLORS_3 = {'Forest': [34,85,34],   'Sparse Vegetation': [255,191,0], 'Bare Soil': [139,69,19]}
CLASS_COLORS_4 = {'Forest': [0,100,0],    'Sparse Vegetation': [255,180,0], 'Bare Soil': [200,0,0],  'Mine': [0,80,200]}
CLASS_TO_INT_4 = {'Forest': 1,            'Sparse Vegetation': 2,           'Bare Soil': 3,          'Mine': 4}
CLASS_TO_INT_3 = {'Forest': 1,            'Sparse Vegetation': 2,           'Bare Soil': 3}
CLASSES_4      = list(CLASS_TO_INT_4.keys())

print('✅ Configuration set')

✅ Configuration set


---
## Step 1 — Load & Preprocess Raw GeoTIFF

**What happens here:**
1. Read the raw 5-band Sentinel-2 GeoTIFF exported from GEE
2. Reproject all bands from EPSG:4326 → **EPSG:32647 (UTM 47N)** at 10 m
3. Bilinear-resample **B11 SWIR** from its native 20 m → 10 m so all bands share the same grid
4. Divide by `SCALE_FACTOR` (10 000) to convert raw DN → **reflectance [0–1]**

In [ ]:
def preprocess_raw_sentinel2(tif_path, band_names, scale_factor,
                              target_crs=TARGET_CRS, target_res=TARGET_RES,
                              swir_bands=SWIR_BANDS):
    """
    Reproject, resample, and scale a raw Sentinel-2 GeoTIFF.

    Parameters
    ----------
    tif_path    : str   — path to raw GeoTIFF
    band_names  : list  — name for each band in the file (same order)
    scale_factor: int   — divide raw DN by this → reflectance (GEE S2 SR = 10 000)
    target_crs  : CRS   — destination CRS (default EPSG:32647)
    target_res  : float — pixel size in metres (default 10 m)
    swir_bands  : set   — band names to bilinear-resample (20 m → 10 m)

    Returns
    -------
    bands_dict  : dict  {band_name: 2D float32 array} — reflectance values
    profile     : dict  — rasterio write profile for UTM grid
    px_area_m2  : float — pixel area in m² (= target_res²)
    H, W        : int   — image height and width after reprojection
    """
    with rasterio.open(tif_path) as src:
        src_crs, src_t = src.crs, src.transform
        nd = src.nodata

        # Compute destination grid
        dst_t, dst_w, dst_h = calculate_default_transform(
            src_crs, target_crs, src.width, src.height,
            *src.bounds, resolution=target_res
        )

        bands_dict = {}
        for bi, bname in enumerate(band_names):
            raw = src.read(bi + 1).astype(np.float32)
            if nd is not None:
                raw[raw == nd] = np.nan

            # Bilinear for SWIR (20 m → 10 m), nearest for 10 m bands
            resamp = Resampling.bilinear if bname in swir_bands else Resampling.nearest
            dst = np.full((dst_h, dst_w), np.nan, dtype=np.float32)
            reproject(
                source=raw, destination=dst,
                src_transform=src_t, src_crs=src_crs,
                dst_transform=dst_t, dst_crs=target_crs,
                resampling=resamp,
                src_nodata=np.nan, dst_nodata=np.nan,
            )
            bands_dict[bname] = (dst / float(scale_factor)).astype(np.float32)

    profile = {
        'driver': 'GTiff', 'dtype': 'float32',
        'crs': target_crs, 'transform': dst_t,
        'width': dst_w, 'height': dst_h,
        'count': len(bands_dict), 'nodata': np.nan,
    }
    px_area_m2 = target_res ** 2   # 100 m²
    return bands_dict, profile, px_area_m2, dst_h, dst_w


print(f'Loading and preprocessing: {INPUT_TIF}')
bands, profile, px_area, H, W = preprocess_raw_sentinel2(
    INPUT_TIF, BAND_NAMES, SCALE_FACTOR
)

print(f'\n✅ Preprocessing complete')
print(f'   Image size : {H} × {W} px  ({H*W/1e6:.2f} M pixels)')
print(f'   CRS        : {profile["crs"]}')
print(f'   Pixel area : {px_area:.0f} m²  ({px_area/10000:.4f} ha)')
print(f'   Bands loaded: {list(bands.keys())}')
for name, arr in bands.items():
    valid = arr[np.isfinite(arr)]
    print(f'   {name:18s}  min={valid.min():.4f}  mean={valid.mean():.4f}  max={valid.max():.4f}')

In [ ]:
# ── RGB True-Colour Preview ─────────────────────────────────────────────────
def make_rgb(bands_dict):
    """Percentile-stretch true-colour composite (B4/B3/B2)."""
    r, g, b = bands_dict['B4_Red'], bands_dict['B3_Green'], bands_dict['B2_Blue']
    def norm(a):
        p2, p98 = np.nanpercentile(a, 2), np.nanpercentile(a, 98)
        return np.clip((a - p2) / (p98 - p2 + 1e-9), 0, 1)
    return np.dstack([norm(r), norm(g), norm(b)])

rgb = make_rgb(bands)

fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(rgb)
ax.set_title('True Colour — Reprojected to UTM 47N (10 m)', fontweight='bold', fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/01_rgb_preview.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 2 — Compute Spectral Indices

| Index | Formula | Target Surface |
|-------|---------|----------------|
| NDVI  | (B8−B4)/(B8+B4) | Dense vegetation / Forest |
| EVI   | 2.5×(B8−B4)/(B8+6B4−7.5B2+1) | Dense forest (less saturation than NDVI) |
| SAVI  | 1.5×(B8−B4)/(B8+B4+0.5) | Sparse vegetation on bare soil |
| NDWI  | (B3−B8)/(B3+B8) | Water bodies & soil moisture |
| BSI   | [(B11+B4)−(B8+B2)] / [(B11+B4)+(B8+B2)] | Bare soil & disturbed land |
| MBI   | (B11+B4−B8)/(B11+B4+B8) | Mine surfaces vs natural bare soil |
| NDBI  | (B11−B8)/(B11+B8) | Built-up / compacted surfaces |
| CMI   | B11/B8 | Clay-rich mine tailings |
| FCI   | B11/B8 | Iron-oxide mine tailings |

In [ ]:
def safe_divide(a, b):
    """Division with NaN for zero/invalid denominators."""
    with np.errstate(invalid='ignore', divide='ignore'):
        return np.where(b != 0, a / b, np.nan).astype(np.float32)


def compute_indices(bands_dict):
    """Compute all 9 spectral indices from a bands_dict."""
    B2  = bands_dict['B2_Blue']
    B3  = bands_dict['B3_Green']
    B4  = bands_dict['B4_Red']
    B8  = bands_dict['B8_NIR']
    B11 = bands_dict['B11_SWIR_10m']

    indices = {}

    # ── Vegetation ──────────────────────────────────────────────────────────
    indices['NDVI'] = safe_divide(B8 - B4, B8 + B4)
    indices['EVI']  = np.clip(
        2.5 * safe_divide(B8 - B4, B8 + 6*B4 - 7.5*B2 + 1), -1, 1
    ).astype(np.float32)
    indices['SAVI'] = safe_divide(1.5 * (B8 - B4), B8 + B4 + 0.5)

    # ── Water ───────────────────────────────────────────────────────────────
    indices['NDWI'] = safe_divide(B3 - B8, B3 + B8)

    # ── Bare Soil / Mine ────────────────────────────────────────────────────
    indices['BSI']  = safe_divide((B11+B4) - (B8+B2), (B11+B4) + (B8+B2))
    indices['MBI']  = safe_divide(B11 + B4 - B8, B11 + B4 + B8)
    indices['NDBI'] = safe_divide(B11 - B8, B11 + B8)

    # ── Geology / Minerals ──────────────────────────────────────────────────
    indices['CMI']  = safe_divide(B11, B8)   # Clay Mineral Index
    indices['FCI']  = safe_divide(B11, B8)   # Ferrous/Iron Oxide Index (proxy; true FCI uses B8A)

    return indices


# Compute and add to bands dict
indices = compute_indices(bands)
bands.update(indices)

print('✅ Indices computed:')
for name, arr in indices.items():
    valid = arr[np.isfinite(arr)]
    print(f'   {name:6s}  min={valid.min():+.4f}  mean={valid.mean():+.4f}  max={valid.max():+.4f}')

In [ ]:
# ── Index Visualisation ─────────────────────────────────────────────────────
SHOW_INDICES = ['NDVI', 'EVI', 'BSI', 'MBI', 'FCI', 'SAVI']
CMAPS        = ['RdYlGn', 'RdYlGn', 'Accent_r', 'hot_r', 'plasma', 'RdYlGn']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, name, cmap in zip(axes.ravel(), SHOW_INDICES, CMAPS):
    im = ax.imshow(bands[name], cmap=cmap, vmin=-0.5, vmax=0.8)
    ax.set_title(name, fontweight='bold', fontsize=11)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle('Spectral Indices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_spectral_indices.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 3 — Feature Selection & Standardisation

Drops correlated/redundant bands and scales to zero mean, unit variance.

**Recommended set:** `B8_NIR, B11_SWIR_10m, NDVI, EVI, SAVI, BSI, FCI`  
Drops: B2/B3/B4 (raw bands redundant with NDVI), NDBI/CMI (correlated with BSI), NDWI/MBI (correlated).

In [ ]:
# ── Build feature matrix ────────────────────────────────────────────────────
print(f'Selected features: {SELECTED_FEATURES}')

stack      = np.stack([bands[f] for f in SELECTED_FEATURES], axis=0)   # (n_feat, H, W)
X_flat     = stack.reshape(len(SELECTED_FEATURES), -1).T                # (H*W, n_feat)
valid_mask = np.all(np.isfinite(X_flat), axis=1)                        # exclude NaN pixels
X_valid    = X_flat[valid_mask]

print(f'Total pixels   : {H*W:,}')
print(f'Valid pixels   : {valid_mask.sum():,}  ({valid_mask.mean()*100:.1f}%)')
print(f'NaN / no-data  : {(~valid_mask).sum():,}')

# ── Standardise ─────────────────────────────────────────────────────────────
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_valid)

print(f'\nFeature statistics (after scaling):')
df_stats = pd.DataFrame(
    {'Feature': SELECTED_FEATURES,
     'Raw mean':  np.nanmean(X_valid, axis=0).round(4),
     'Raw std':   np.nanstd(X_valid,  axis=0).round(4),
     'Scaled mean': X_scaled.mean(axis=0).round(4),
     'Scaled std':  X_scaled.std(axis=0).round(4)}
)
print(df_stats.to_string(index=False))
print('\n✅ Feature matrix ready')

---
## Step 4 — K-Means Unsupervised Clustering

Fits K-Means on a **50 000-pixel subsample** for speed, then predicts on the full image.
After clustering, assign a land-cover class to each cluster by inspecting centroid NDVI / BSI values:
- Highest NDVI → **Forest**
- Moderate NDVI → **Sparse Vegetation**  
- Lowest NDVI + highest BSI → **Bare Soil**

Post-classification cleanup: pixels with NDVI ≥ 0.75 are reclassified as Forest.

In [ ]:
# ── Fit K-Means on subsample ────────────────────────────────────────────────
MAX_KM_PIXELS = 50_000
rng = np.random.default_rng(RANDOM_STATE)

idx_sub = rng.choice(len(X_scaled), min(MAX_KM_PIXELS, len(X_scaled)), replace=False)
X_sub   = X_scaled[idx_sub]

print(f'Fitting K-Means (k={N_CLUSTERS}) on {len(X_sub):,} subsample pixels…')
km = KMeans(n_clusters=N_CLUSTERS, init='k-means++', n_init=3,
            max_iter=300, random_state=RANDOM_STATE)
km.fit(X_sub)
print('K-Means fitted.')

# ── Predict on full image in batches ───────────────────────────────────────
print('Predicting full image…')
BATCH   = 500_000
km_labels = np.empty(len(X_scaled), dtype=np.int32)
for s in range(0, len(X_scaled), BATCH):
    km_labels[s:s+BATCH] = km.predict(X_scaled[s:s+BATCH])

# Silhouette (subsample)
idx_sil = rng.choice(len(X_scaled), min(20_000, len(X_scaled)), replace=False)
sil     = silhouette_score(X_scaled[idx_sil], km_labels[idx_sil])
print(f'\nSilhouette Score : {sil:.4f}')

# ── Centroid table ──────────────────────────────────────────────────────────
centroids = scaler.inverse_transform(km.cluster_centers_)
df_cents  = pd.DataFrame(centroids, columns=SELECTED_FEATURES,
                         index=[f'Cluster {i}' for i in range(N_CLUSTERS)])
print('\nCluster Centroids (reflectance / index values):')
print(df_cents.round(4).to_string())

In [ ]:
# ── MANUAL STEP: assign class labels to clusters ────────────────────────────
#
#  Inspect the centroids above and set CLUSTER_LABELS accordingly.
#  Rule of thumb:
#    • Highest NDVI centroid    → 'Forest'
#    • Moderate NDVI centroid   → 'Sparse Vegetation'
#    • Lowest NDVI / high BSI   → 'Bare Soil'
#    • If k=4: assign two clusters the same name where they overlap (e.g. both 'Forest')
#
#  EDIT THIS DICT to match your cluster indices:
CLUSTER_LABELS = {
    0: 'Forest',
    1: 'Bare Soil',
    2: 'Sparse Vegetation',
    3: 'Forest',       # second forest cluster (typical for k=4)
}

print('Assigned labels:')
for k, v in CLUSTER_LABELS.items():
    ndvi = df_cents.loc[f'Cluster {k}', 'NDVI'] if 'NDVI' in df_cents.columns else 'n/a'
    bsi  = df_cents.loc[f'Cluster {k}', 'BSI']  if 'BSI'  in df_cents.columns else 'n/a'
    print(f'  Cluster {k} → {v:20s}  NDVI={ndvi:.4f}  BSI={bsi:.4f}')

In [ ]:
# ── Build label image ───────────────────────────────────────────────────────
km_label_img = np.full(H * W, -1, dtype=np.int16)
km_label_img[valid_mask] = km_labels
km_label_img = km_label_img.reshape(H, W)

# ── NDVI cleanup: NDVI >= 0.75 → Forest ───────────────────────────────────
ndvi_band   = bands['NDVI']
forest_cids = [cid for cid, name in CLUSTER_LABELS.items() if name == 'Forest']
for cid in range(N_CLUSTERS):
    if CLUSTER_LABELS[cid] != 'Forest':
        # Pixels wrongly not-Forest but with high NDVI → reassign to first Forest cluster
        km_label_img[(ndvi_band >= NDVI_CLEANUP) & (km_label_img == cid)] = forest_cids[0]

print(f'✅ NDVI cleanup applied (threshold = {NDVI_CLEANUP})')

# ── Area statistics ─────────────────────────────────────────────────────────
from collections import Counter
counts = Counter(km_label_img[km_label_img >= 0].ravel().tolist())
rows = []
for cid in range(N_CLUSTERS):
    name = CLUSTER_LABELS[cid]
    n    = counts.get(cid, 0)
    rows.append({'Cluster': cid, 'Class': name,
                 'Pixels': n, 'Hectares': round(n * px_area / 10_000, 1),
                 'Coverage %': round(n / valid_mask.sum() * 100, 1)})
df_km = pd.DataFrame(rows)
print('\nK-Means Area Summary:')
print(df_km.to_string(index=False))

In [ ]:
# ── Visualise K-Means result ────────────────────────────────────────────────
unique_cids   = sorted(CLUSTER_LABELS.keys())
cmap_colors   = [np.array(CLASS_COLORS_3.get(CLUSTER_LABELS[c], [128,128,128]))/255
                 for c in unique_cids]
cmap_km       = ListedColormap(cmap_colors)
patch_labels  = sorted(set(CLUSTER_LABELS.values()))
patches       = [mpatches.Patch(color=np.array(CLASS_COLORS_3[n])/255, label=n)
                 for n in patch_labels]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
ax1.imshow(rgb); ax1.set_title('True Colour', fontweight='bold'); ax1.axis('off')
ax2.imshow(km_label_img, cmap=cmap_km, vmin=0, vmax=N_CLUSTERS - 1)
ax2.set_title(f'K-Means (k={N_CLUSTERS}) — after NDVI cleanup', fontweight='bold')
ax2.axis('off')
ax2.legend(handles=patches, loc='lower right', fontsize=10, framealpha=0.9)

plt.suptitle('K-Means Classification', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/03_kmeans_classification.png', dpi=150, bbox_inches='tight')
plt.show()

# ── PCA scatter ─────────────────────────────────────────────────────────────
idx_p = rng.choice(len(X_scaled), min(10_000, len(X_scaled)), replace=False)
pca   = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d  = pca.fit_transform(X_scaled[idx_p])
c2d   = pca.transform(km.cluster_centers_)

fig, ax = plt.subplots(figsize=(9, 6))
for cid in unique_cids:
    m   = km_labels[idx_p] == cid
    col = cmap_colors[cid]
    ax.scatter(X_2d[m, 0], X_2d[m, 1], s=6, alpha=0.4, color=col, linewidths=0,
               label=CLUSTER_LABELS[cid])
ax.scatter(c2d[:,0], c2d[:,1], s=220, marker='X', color='white',
           edgecolors='black', linewidths=1.5, zorder=5, label='Centroids')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('K-Means Clusters — PCA Projection', fontweight='bold')
ax.legend(markerscale=2, fontsize=9); ax.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/03b_kmeans_pca_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 5 — 4-Class Mine RF

Adds a **Mine** class by rasterizing a polygon shapefile and sampling pixels inside it.

**Mine training mask:** inside polygon AND NDVI < 0.55 AND NDWI > −0.22  
Spectral filtering removes vegetated polygon edges and non-mine land.

In [ ]:
# ── Load and rasterize mine shapefile ───────────────────────────────────────
print(f'Loading mine shapefile: {MINE_SHP}')
gdf_mine = gpd.read_file('../Mine_Class_USGS/Final_mines_polygons/ree_mines_roi.shp')
print(f'  {len(gdf_mine)} polygon(s) | CRS: {gdf_mine.crs}')

# Reproject to match raster CRS
gdf_mine = gdf_mine.to_crs(profile['crs'])

mine_mask = rasterize(
    [(mapping(geom), 1) for geom in gdf_mine.geometry],
    out_shape=(H, W),
    transform=profile['transform'],   # UTM 47N reprojected transform
    fill=0,
    dtype=np.uint8
)

n_mine_px = mine_mask.sum()
print(f'  Rasterized → {n_mine_px:,} mine pixels (~{n_mine_px*px_area/10_000:,.1f} ha)')

# ── Spectral profile of mine pixels (diagnostic) ────────────────────────────
mine_flat  = mine_mask.ravel().astype(bool)
diag_bands = ['NDVI', 'NDWI', 'BSI', 'FCI', 'B8_NIR', 'B11_SWIR_10m']
rows_d = []
for bn in diag_bands:
    vals = bands[bn].ravel()[mine_flat]
    vals = vals[np.isfinite(vals)]
    rows_d.append({'Band': bn, 'Min': vals.min().round(4),
                   'Mean': vals.mean().round(4), 'Max': vals.max().round(4)})
print('\nMine pixel spectral profile:')
print(pd.DataFrame(rows_d).to_string(index=False))

In [ ]:
# ── Build training masks ─────────────────────────────────────────────────────
#
#   Forest     : K-Means label_img == Forest cluster  AND  NDVI >= FOREST_NDVI_MIN (0.70)
#   Sparse Veg : K-Means label_img == Sparse Vegetation cluster  (no extra threshold)
#   Bare Soil  : K-Means label_img == Bare Soil cluster           (no extra threshold)
#   Mine       : mine_mask AND NDVI < MINE_NDVI_MAX AND NDWI > MINE_NDWI_MIN
#
# All masks are 2-D (H, W) — same coordinate space as km_label_img and bands.
# X_full covers all H*W pixels (NaN rows filtered inside safe_sample).
# This matches satellite_classifier_raw.py exactly.

def get_cluster_ids(label_name):
    return [cid for cid, name in CLUSTER_LABELS.items() if name == label_name]

f_cids = get_cluster_ids('Forest')
s_cids = get_cluster_ids('Sparse Vegetation')
b_cids = get_cluster_ids('Bare Soil')

# 2-D masks over the full image (H, W)
f_mask     = np.isin(km_label_img, f_cids) & (bands['NDVI'] >= FOREST_NDVI_MIN)
s_mask     = np.isin(km_label_img, s_cids)
b_mask     = np.isin(km_label_img, b_cids)
mine_train = mine_mask.astype(bool) \
             & (bands['NDVI'] < MINE_NDVI_MAX) \
             & (bands['NDWI'] > MINE_NDWI_MIN)

# Full H*W feature matrix (unscaled) — NaN rows filtered inside safe_sample
X_full = X_flat.copy()   # shape (H*W, n_feat)

def safe_sample(mask_2d, X_full, n, rng):
    """Sample n pixels from a 2-D boolean mask into raw feature rows."""
    idx = np.where(mask_2d.ravel())[0]
    idx = idx[~np.any(np.isnan(X_full[idx]), axis=1)]
    chosen = rng.choice(len(idx), min(n, len(idx)), replace=False)
    return X_full[idx[chosen]], idx[chosen]

print('Training mask counts:')
print(f'  Forest            : {f_mask.sum():>8,} px  (K-Means + NDVI >= {FOREST_NDVI_MIN})')
print(f'  Sparse Vegetation : {s_mask.sum():>8,} px  (K-Means)')
print(f'  Bare Soil         : {b_mask.sum():>8,} px  (K-Means)')
print(f'  Mine (filtered)   : {mine_train.sum():>8,} px')

rng_m = np.random.default_rng(RANDOM_STATE)
X_f4, _ = safe_sample(f_mask,     X_full, N_SAMPLES_PER_CLS, rng_m)
X_s4, _ = safe_sample(s_mask,     X_full, N_SAMPLES_PER_CLS, rng_m)
X_b4, _ = safe_sample(b_mask,     X_full, N_SAMPLES_PER_CLS, rng_m)
X_m4, _ = safe_sample(mine_train, X_full, N_SAMPLES_PER_CLS, rng_m)

print(f'\nSamples — Forest: {len(X_f4):,} | Sparse: {len(X_s4):,} | '
      f'Bare: {len(X_b4):,} | Mine: {len(X_m4):,}')

X_train_4_raw = np.concatenate([X_f4, X_s4, X_b4, X_m4])
y_train_4     = np.array(
    [CLASS_TO_INT_4['Forest']]            * len(X_f4) +
    [CLASS_TO_INT_4['Sparse Vegetation']] * len(X_s4) +
    [CLASS_TO_INT_4['Bare Soil']]         * len(X_b4) +
    [CLASS_TO_INT_4['Mine']]              * len(X_m4), dtype=np.int32
)

# Scale using the SAME scaler fitted in Step 3
X_train_4_sc = scaler.transform(X_train_4_raw)

# ── Train 4-class RF ────────────────────────────────────────────────────────
X_tr4, X_te4, y_tr4, y_te4 = train_test_split(
    X_train_4_sc, y_train_4, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train_4)

print(f'\nTraining 4-class Mine RF (n_estimators={N_ESTIMATORS})...')
rf4 = RandomForestClassifier(
    n_estimators=N_ESTIMATORS, class_weight='balanced',
    n_jobs=-1, random_state=RANDOM_STATE, oob_score=True
)
rf4.fit(X_tr4, y_tr4)
y_pred4 = rf4.predict(X_te4)
print(f'\n✅ OOB Accuracy  : {rf4.oob_score_:.4f}')
print(f'   Test Accuracy : {(y_pred4==y_te4).mean():.4f}')


In [ ]:
# ── Predict full image ──────────────────────────────────────────────────────
print('Predicting full image…')
rf4_labels = np.zeros(H * W, dtype=np.uint8)
for s in range(0, len(X_scaled), BATCH):
    idx_v = np.where(valid_mask)[0][s:s+BATCH]
    rf4_labels[idx_v] = rf4.predict(X_scaled[s:s+BATCH]).astype(np.uint8)
rf4_img = rf4_labels.reshape(H, W)

# ── Area statistics ─────────────────────────────────────────────────────────
rows4 = []
total_valid = valid_mask.sum()
for name, lbl in CLASS_TO_INT_4.items():
    n = int((rf4_img == lbl).sum())
    rows4.append({'Class': name, 'Pixels': n,
                  'Hectares': round(n * px_area / 10_000, 1),
                  'Coverage %': round(n / total_valid * 100, 1)})
df_rf4 = pd.DataFrame(rows4)
print('\n4-Class Mine RF Area Summary:')
print(df_rf4.to_string(index=False))

In [ ]:
# ── Visualise 4-Class Mine RF ───────────────────────────────────────────────
color_map_int = {0: [0,0,0], **{CLASS_TO_INT_4[k]: v for k,v in CLASS_COLORS_4.items()}}

def build_class_rgb(img_int, color_map):
    h, w = img_int.shape
    rgb_out = np.zeros((h, w, 3), dtype=np.uint8)
    for lbl, col in color_map.items():
        rgb_out[img_int == lbl] = col
    return rgb_out

rgb_cls4  = build_class_rgb(rf4_img, color_map_int)
patches4  = [mpatches.Patch(color=np.array(CLASS_COLORS_4[c])/255, label=c) for c in CLASSES_4]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 9), gridspec_kw={'wspace': 0.02})
ax1.imshow(rgb,     aspect='auto'); ax1.set_title('True Colour',      fontweight='bold', fontsize=12); ax1.axis('off')
ax2.imshow(rgb_cls4,aspect='auto'); ax2.set_title('4-Class Mine RF', fontweight='bold', fontsize=12); ax2.axis('off')
ax2.legend(handles=patches4, loc='lower left', fontsize=10, framealpha=0.9, title='Land Cover')

plt.suptitle('Land Cover Classification — Kachin State REE Sites', fontsize=14, fontweight='bold')
plt.savefig(f'{OUTPUT_DIR}/05_4class_mine_rf.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Area bar chart ───────────────────────────────────────────────────────────
fig, (ax_bar, ax_fi) = plt.subplots(1, 2, figsize=(14, 5))
areas = [df_rf4.loc[df_rf4['Class']==c, 'Hectares'].values[0] for c in CLASSES_4]
bars  = ax_bar.barh(CLASSES_4, areas,
                    color=[np.array(CLASS_COLORS_4[c])/255 for c in CLASSES_4],
                    edgecolor='white')
for bar, area in zip(bars, areas):
    ax_bar.text(bar.get_width() + max(areas)*0.01, bar.get_y() + bar.get_height()/2,
                f'{area:,.0f} ha', va='center', fontsize=10)
ax_bar.set_xlabel('Area (Hectares)'); ax_bar.set_title('Class Area Breakdown', fontweight='bold')
ax_bar.spines['top'].set_visible(False); ax_bar.spines['right'].set_visible(False)

imp4  = rf4.feature_importances_; sidx4 = np.argsort(imp4)[::-1]
ax_fi.bar([SELECTED_FEATURES[i] for i in sidx4], imp4[sidx4], color='#378ADD', edgecolor='white')
ax_fi.set_ylabel('Gini Importance'); ax_fi.set_title('Feature Importance', fontweight='bold')
ax_fi.tick_params(axis='x', rotation=30)
ax_fi.spines['top'].set_visible(False); ax_fi.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/05b_4class_mine_stats.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Confusion matrix ─────────────────────────────────────────────────────────
INT_TO_CLS_4 = {v: k for k, v in CLASS_TO_INT_4.items()}
ul4    = sorted(np.unique(y_te4))
unames4 = [INT_TO_CLS_4[i] for i in ul4]

fig, ax_cm = plt.subplots(figsize=(6, 5))
cm4 = confusion_matrix(y_te4, y_pred4, labels=ul4)
im  = ax_cm.imshow(cm4, cmap='Blues'); fig.colorbar(im, ax=ax_cm, fraction=0.046, pad=0.04)
ax_cm.set_xticks(range(len(ul4))); ax_cm.set_xticklabels(unames4, rotation=30, ha='right')
ax_cm.set_yticks(range(len(ul4))); ax_cm.set_yticklabels(unames4)
for i in range(cm4.shape[0]):
    for j in range(cm4.shape[1]):
        ax_cm.text(j, i, format(cm4[i,j],'d'), ha='center', va='center',
                   color='white' if cm4[i,j] > cm4.max()/2 else 'black', fontweight='bold')
ax_cm.set_title(f'4-Class Confusion Matrix (OOB: {rf4.oob_score_:.4f})', fontweight='bold')
ax_cm.set_xlabel('Predicted'); ax_cm.set_ylabel('True')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/05c_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(classification_report(y_te4, y_pred4, labels=ul4, target_names=unames4))

In [ ]:
# ── Save models ─────────────────────────────────────────────────────────────
with open(f'{OUTPUT_DIR}/kmeans_model.pkl',       'wb') as f: pickle.dump(km,     f)
with open(f'{OUTPUT_DIR}/rf_4class_mine_model.pkl','wb') as f: pickle.dump(rf4,   f)
with open(f'{OUTPUT_DIR}/feature_scaler.pkl',     'wb') as f: pickle.dump(scaler, f)

with open(f'{OUTPUT_DIR}/selected_features.txt', 'w') as f:
    f.write(','.join(SELECTED_FEATURES))

print('✅ Models saved to:', OUTPUT_DIR)
for fname in os.listdir(OUTPUT_DIR):
    if fname.endswith('.pkl') or fname.endswith('.txt'):
        size = os.path.getsize(f'{OUTPUT_DIR}/{fname}')
        print(f'   {fname:35s}  {size/1024:.1f} KB')

---
## Step 6 — Multi-Year Change Analysis

Applies the trained 4-class RF model to each yearly raw GeoTIFF.
Each image goes through the **full pipeline**: reproject → scale → compute indices → classify.

Produces:
- Annual area table (Forest, Sparse Vegetation, Bare Soil, Mine)
- Stacked area chart over time
- Forest loss & mine growth trend lines
- Annual rate-of-change bar chart
- Key metrics relative to baseline year

In [ ]:
# ── Classify each year ──────────────────────────────────────────────────────
yearly_results = {}

for year, tif_path in sorted(YEARLY_TIFS.items()):
    if not os.path.exists(tif_path):
        print(f'  ⚠️  {year}: file not found — {tif_path}')
        continue

    print(f'  Classifying {year}…', end=' ')

    # Full pipeline per year
    bd_yr, _, pxa_yr, h_yr, w_yr = preprocess_raw_sentinel2(
        tif_path, BAND_NAMES, SCALE_FACTOR)
    idx_yr = compute_indices(bd_yr)
    bd_yr.update(idx_yr)

    # Feature matrix
    stack_yr  = np.stack([bd_yr[f] for f in SELECTED_FEATURES], axis=0)
    Xf_yr     = stack_yr.reshape(len(SELECTED_FEATURES), -1).T
    valid_yr  = np.all(np.isfinite(Xf_yr), axis=1)
    Xsc_yr    = np.full_like(Xf_yr, np.nan)
    Xsc_yr[valid_yr] = scaler.transform(Xf_yr[valid_yr])

    # Batched prediction
    labels_yr = np.zeros(h_yr * w_yr, dtype=np.uint8)
    for s in range(0, valid_yr.sum(), BATCH):
        idx_v = np.where(valid_yr)[0][s:s+BATCH]
        labels_yr[idx_v] = rf4.predict(Xsc_yr[idx_v]).astype(np.uint8)
    img_yr = labels_yr.reshape(h_yr, w_yr)

    areas_yr = {n: int((img_yr == v).sum()) * pxa_yr / 10_000
                for n, v in CLASS_TO_INT_4.items()}
    yearly_results[year] = {'image': img_yr, 'areas': areas_yr,
                             'bands_dict': bd_yr, 'h': h_yr, 'w': w_yr}

    f_ha = areas_yr['Forest']; m_ha = areas_yr['Mine']
    print(f'Forest={f_ha:,.0f} ha  Mine={m_ha:,.0f} ha')

print(f'\n✅ Classified {len(yearly_results)} year(s)')

In [ ]:
# ── Shared setup ────────────────────────────────────────────────────────────
years_sorted = sorted(yearly_results.keys())
base         = BASELINE_YEAR if BASELINE_YEAR in yearly_results else years_sorted[0]
last_yr      = years_sorted[-1]
span         = last_yr - years_sorted[0] if len(years_sorted) > 1 else 1

forest_ha = [yearly_results[y]['areas']['Forest']            for y in years_sorted]
sparse_ha = [yearly_results[y]['areas']['Sparse Vegetation'] for y in years_sorted]
bare_ha   = [yearly_results[y]['areas']['Bare Soil']         for y in years_sorted]
mine_ha   = [yearly_results[y]['areas']['Mine']              for y in years_sorted]

bf  = yearly_results[base]['areas']['Forest']
bsv = yearly_results[base]['areas']['Sparse Vegetation']
bbs = yearly_results[base]['areas']['Bare Soil']
bm  = yearly_results[base]['areas']['Mine']

total_valid_ha = {y: sum(yearly_results[y]['areas'].values()) for y in years_sorted}

def pct_of_total(ha, yr):
    return ha / total_valid_ha[yr] * 100 if total_valid_ha[yr] > 0 else 0

def pct_chg(v, base_v):
    return (v - base_v) / base_v * 100 if base_v > 0 else 0

total_fl = bf - yearly_results[last_yr]['areas']['Forest']
total_mg = yearly_results[last_yr]['areas']['Mine'] - bm

# ── Overview: key metrics ────────────────────────────────────────────────────
print('📊 Key Metrics')
print(f'   Baseline year        : {base}')
print(f'   Analysis period      : {years_sorted[0]}–{last_yr} ({span} yr)')
print(f'   Total forest loss    : {total_fl:,.0f} ha  ({total_fl/bf*100:.1f}% of baseline)')
print(f'   Avg deforestation    : {total_fl/span:,.0f} ha/yr')
print(f'   Total mine growth    : {total_mg:,.0f} ha  ({pct_chg(yearly_results[last_yr]["areas"]["Mine"], bm):.1f}% of baseline)' if bm > 0 else f'   Total mine growth    : {total_mg:,.0f} ha  (new area)')
print(f'   Avg mine expansion   : {total_mg/span:,.0f} ha/yr')
print(f'   Forest (baseline)    : {bf:,.0f} ha  ({pct_of_total(bf, base):.1f}% of study area)')
print(f'   Mine   (baseline)    : {bm:,.0f} ha  ({pct_of_total(bm, base):.1f}% of study area)')
print(f'   Forest (latest)      : {yearly_results[last_yr]["areas"]["Forest"]:,.0f} ha  ({pct_of_total(yearly_results[last_yr]["areas"]["Forest"], last_yr):.1f}% of study area)')
print(f'   Mine   (latest)      : {yearly_results[last_yr]["areas"]["Mine"]:,.0f} ha  ({pct_of_total(yearly_results[last_yr]["areas"]["Mine"], last_yr):.1f}% of study area)')

# ── Full summary table with % coverage ──────────────────────────────────────
rows_sum = []
for y in years_sorted:
    a  = yearly_results[y]['areas']
    rows_sum.append({
        'Year':                        y,
        'Forest (ha)':                 f"{a['Forest']:,.0f}",
        'Forest (%)':                  f"{pct_of_total(a['Forest'], y):.1f}%",
        'Sparse Veg (ha)':             f"{a['Sparse Vegetation']:,.0f}",
        'Sparse Veg (%)':              f"{pct_of_total(a['Sparse Vegetation'], y):.1f}%",
        'Bare Soil (ha)':              f"{a['Bare Soil']:,.0f}",
        'Bare Soil (%)':               f"{pct_of_total(a['Bare Soil'], y):.1f}%",
        'Mine (ha)':                   f"{a['Mine']:,.0f}",
        'Mine (%)':                    f"{pct_of_total(a['Mine'], y):.1f}%",
        f'Forest Δ vs {base}':         f"{pct_chg(a['Forest'], bf):+.1f}%",
        f'Mine Δ vs {base}':           f"{pct_chg(a['Mine'], bm):+.1f}%" if bm > 0 else 'n/a',
    })
df_sum = pd.DataFrame(rows_sum)
print('\nAnnual Land Cover Summary:')
print(df_sum.to_string(index=False))

# Save full numeric CSV
csv_rows = []
for y in years_sorted:
    a = yearly_results[y]['areas']
    csv_rows.append({
        'Year':            y,
        'Forest_ha':       a['Forest'],
        'Forest_pct':      pct_of_total(a['Forest'], y),
        'SparseVeg_ha':    a['Sparse Vegetation'],
        'SparseVeg_pct':   pct_of_total(a['Sparse Vegetation'], y),
        'BareSoil_ha':     a['Bare Soil'],
        'BareSoil_pct':    pct_of_total(a['Bare Soil'], y),
        'Mine_ha':         a['Mine'],
        'Mine_pct':        pct_of_total(a['Mine'], y),
        'Forest_delta_pct': pct_chg(a['Forest'], bf),
        'Mine_delta_pct':   pct_chg(a['Mine'], bm) if bm > 0 else None,
    })
pd.DataFrame(csv_rows).to_csv(f'{OUTPUT_DIR}/multi_year_summary.csv', index=False)
print(f'\n✅ Saved: {OUTPUT_DIR}/multi_year_summary.csv')


In [ ]:
# ── Classification Maps ──────────────────────────────────────────────────────
# Per-year: true colour (left) + 4-class RF (right)
color_map_int = {0: [0,0,0], **{CLASS_TO_INT_4[k]: v for k, v in CLASS_COLORS_4.items()}}
legend_patches = [mpatches.Patch(color=np.array(CLASS_COLORS_4[c])/255, label=c) for c in CLASSES_4] +                  [mpatches.Patch(color='black', label='No Data')]

def downsample(arr, max_px=600):
    h, w = arr.shape[:2]
    f = max(1, max(h, w) // max_px)
    return arr[::f, ::f]

def make_rgb_img(bands_dict):
    r, g, b = bands_dict.get('B4_Red'), bands_dict.get('B3_Green'), bands_dict.get('B2_Blue')
    if r is None or g is None or b is None:
        return None
    def norm(a):
        p2, p98 = np.nanpercentile(a, 2), np.nanpercentile(a, 98)
        return np.clip((a - p2) / (p98 - p2 + 1e-9), 0, 1)
    return np.dstack([norm(r), norm(g), norm(b)])

for yr in years_sorted:
    res_yr   = yearly_results[yr]
    h_yr, w_yr = res_yr['h'], res_yr['w']
    rgb_yr   = make_rgb_img(res_yr['bands_dict'])
    cls_rgb  = build_class_rgb(res_yr['image'], color_map_int)

    a = res_yr['areas']
    print(f"{yr}  Forest: {a['Forest']:,.0f} ha ({pct_of_total(a['Forest'],yr):.1f}%) | "
          f"Sparse Veg: {a['Sparse Vegetation']:,.0f} ha ({pct_of_total(a['Sparse Vegetation'],yr):.1f}%) | "
          f"Bare Soil: {a['Bare Soil']:,.0f} ha ({pct_of_total(a['Bare Soil'],yr):.1f}%) | "
          f"Mine: {a['Mine']:,.0f} ha ({pct_of_total(a['Mine'],yr):.1f}%)")

    fig_yr, (ax_tc, ax_cl) = plt.subplots(1, 2, figsize=(16, 5),
                                            gridspec_kw={'wspace': 0.02})
    fig_yr.suptitle(f'Land Cover Classification — {yr}', fontsize=12, fontweight='bold')
    if rgb_yr is not None:
        ax_tc.imshow(downsample(rgb_yr))
        ax_tc.set_title('True Colour (B4/B3/B2)', fontweight='bold')
    else:
        ax_tc.set_title('True Colour (unavailable)')
        ax_tc.text(0.5, 0.5, 'No RGB', ha='center', transform=ax_tc.transAxes)
    ax_tc.axis('off')
    ax_cl.imshow(downsample(cls_rgb))
    ax_cl.set_title('4-Class RF Classification', fontweight='bold')
    ax_cl.axis('off')
    ax_cl.legend(handles=legend_patches, loc='lower left', fontsize=8,
                 framealpha=0.85, title='Land Cover', title_fontsize=8)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/05_map_{yr}.png', dpi=150, bbox_inches='tight')
    plt.show(); plt.close()

# ── First vs last year 2×2 comparison ───────────────────────────────────────
if len(years_sorted) >= 2:
    fig_cmp, axes_cmp = plt.subplots(2, 2, figsize=(18, 12),
                                      gridspec_kw={'wspace': 0.02, 'hspace': 0.12})
    fig_cmp.suptitle(f'Land Cover Change: {years_sorted[0]} → {last_yr}',
                     fontsize=13, fontweight='bold')
    for col_i, yr_c in enumerate([years_sorted[0], last_yr]):
        res_c = yearly_results[yr_c]
        rgb_c = make_rgb_img(res_c['bands_dict'])
        cls_c = build_class_rgb(res_c['image'], color_map_int)
        if rgb_c is not None:
            axes_cmp[0, col_i].imshow(downsample(rgb_c))
        axes_cmp[0, col_i].set_title(f'{yr_c} — True Colour', fontweight='bold')
        axes_cmp[0, col_i].axis('off')
        axes_cmp[1, col_i].imshow(downsample(cls_c))
        axes_cmp[1, col_i].set_title(f'{yr_c} — Classification', fontweight='bold')
        axes_cmp[1, col_i].axis('off')
    axes_cmp[1, 1].legend(handles=legend_patches, loc='lower left', fontsize=9,
                          framealpha=0.85, title='Land Cover')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/05b_first_vs_last.png', dpi=150, bbox_inches='tight')
    plt.show(); plt.close()


In [ ]:
# ── Trend Charts ─────────────────────────────────────────────────────────────

# 1. Stacked area ─────────────────────────────────────────────────────────────
fig_sa, ax_sa = plt.subplots(figsize=(12, 5))
ax_sa.stackplot(years_sorted, forest_ha, sparse_ha, bare_ha, mine_ha,
                labels=CLASSES_4,
                colors=[np.array(CLASS_COLORS_4[c])/255 for c in CLASSES_4], alpha=0.85)
ax_sa.set_xlabel('Year'); ax_sa.set_ylabel('Area (ha)')
ax_sa.set_title('Land Cover Area Over Time', fontweight='bold')
ax_sa.legend(loc='upper right', fontsize=9)
ax_sa.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax_sa.spines['top'].set_visible(False); ax_sa.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/06a_trend_stacked_area.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

# 2. % composition stacked bar per year ───────────────────────────────────────
fig_pct, ax_pct = plt.subplots(figsize=(12, 5))
bottoms = np.zeros(len(years_sorted))
for cls_n in CLASSES_4:
    pcts = [pct_of_total(yearly_results[y]['areas'][cls_n], y) for y in years_sorted]
    ax_pct.bar(years_sorted, pcts, bottom=bottoms,
               color=np.array(CLASS_COLORS_4[cls_n])/255,
               label=cls_n, edgecolor='white', width=0.5)
    for xi, (ys, pv, bv) in enumerate(zip(years_sorted, pcts, bottoms)):
        if pv >= 3:
            ax_pct.text(ys, bv + pv / 2, f'{pv:.1f}%',
                        ha='center', va='center', fontsize=8, color='white', fontweight='bold')
    bottoms += np.array(pcts)
ax_pct.set_ylabel('% of Study Area'); ax_pct.set_ylim(0, 100)
ax_pct.set_title('Land Cover Composition (%) by Year', fontweight='bold')
ax_pct.legend(loc='lower right', fontsize=9)
ax_pct.spines['top'].set_visible(False); ax_pct.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/06b_trend_pct_composition.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

# 3. Forest vs Mine dual-axis with point annotations ─────────────────────────
fig_fm, ax_f = plt.subplots(figsize=(12, 5))
ax_m = ax_f.twinx()
l1, = ax_f.plot(years_sorted, forest_ha, 'o-',
                color=np.array(CLASS_COLORS_4['Forest'])/255,
                linewidth=2.5, markersize=8, label='Forest')
l2, = ax_m.plot(years_sorted, mine_ha, 's--',
                color=np.array(CLASS_COLORS_4['Mine'])/255,
                linewidth=2.5, markersize=8, label='Mine')
for y, fv, mv in zip(years_sorted, forest_ha, mine_ha):
    ax_f.annotate(f'{fv:,.0f}', (y, fv), textcoords='offset points',
                  xytext=(0, 8), ha='center', fontsize=8,
                  color=np.array(CLASS_COLORS_4['Forest'])/255)
    ax_m.annotate(f'{mv:,.0f}', (y, mv), textcoords='offset points',
                  xytext=(0, -14), ha='center', fontsize=8,
                  color=np.array(CLASS_COLORS_4['Mine'])/255)
ax_f.set_ylabel('Forest Area (ha)'); ax_m.set_ylabel('Mine Area (ha)')
ax_f.set_title('Forest Cover vs Mine Expansion', fontweight='bold')
ax_f.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax_m.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax_f.legend(handles=[l1, l2], loc='center left', fontsize=9)
ax_f.grid(True, linestyle='--', alpha=0.3); ax_f.spines['top'].set_visible(False)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/06c_trend_forest_vs_mine.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

# 4. Cumulative % change vs baseline — all 4 classes ─────────────────────────
fig_cum, ax_cum = plt.subplots(figsize=(12, 5))
base_vals = {'Forest': bf, 'Sparse Vegetation': bsv, 'Bare Soil': bbs, 'Mine': bm}
for cls_n in CLASSES_4:
    bv = base_vals[cls_n]
    if bv > 0:
        cpcts = [pct_chg(yearly_results[y]['areas'][cls_n], bv) for y in years_sorted]
        ax_cum.plot(years_sorted, cpcts, 'o-',
                    color=np.array(CLASS_COLORS_4[cls_n])/255,
                    linewidth=2, markersize=7, label=cls_n)
        for y, cp in zip(years_sorted, cpcts):
            if y != base:
                ax_cum.annotate(f'{cp:+.1f}%', (y, cp),
                                textcoords='offset points', xytext=(0, 7),
                                ha='center', fontsize=8)
ax_cum.axhline(0, color='grey', linewidth=1, linestyle='--')
ax_cum.set_ylabel(f'Change vs {base} (%)'); ax_cum.set_xlabel('Year')
ax_cum.set_title(f'Cumulative % Change from {base} Baseline', fontweight='bold')
ax_cum.legend(fontsize=9)
ax_cum.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:+.1f}%'))
ax_cum.grid(True, linestyle='--', alpha=0.3)
ax_cum.spines['top'].set_visible(False); ax_cum.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/06d_trend_cumulative_pct.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()


In [ ]:
# ── Change Rates ─────────────────────────────────────────────────────────────
if len(years_sorted) >= 2:
    yr_labels   = []
    defor_rate  = []; mine_rate  = []
    sparse_rate = []; bare_rate  = []
    rows_rate   = []

    for y0, y1 in zip(years_sorted[:-1], years_sorted[1:]):
        gap   = y1 - y0
        label = f'{y0}–{y1}'
        yr_labels.append(label)
        f0, f1 = yearly_results[y0]['areas']['Forest'],            yearly_results[y1]['areas']['Forest']
        s0, s1 = yearly_results[y0]['areas']['Sparse Vegetation'], yearly_results[y1]['areas']['Sparse Vegetation']
        b0, b1 = yearly_results[y0]['areas']['Bare Soil'],         yearly_results[y1]['areas']['Bare Soil']
        m0, m1 = yearly_results[y0]['areas']['Mine'],              yearly_results[y1]['areas']['Mine']
        dr = (f0 - f1) / gap; mr = (m1 - m0) / gap
        sr = (s1 - s0) / gap; br = (b1 - b0) / gap
        defor_rate.append(dr); mine_rate.append(mr)
        sparse_rate.append(sr); bare_rate.append(br)
        rows_rate.append({
            'Period':                  label,
            'Forest Loss (ha/yr)':     f'{dr:+,.1f}',
            'Forest Loss (%)':         f'{pct_chg(f1, f0):+.1f}%',
            'Mine Growth (ha/yr)':     f'{mr:+,.1f}',
            'Mine Growth (%)':         f'{pct_chg(m1, m0):+.1f}%' if m0 > 0 else 'new',
            'Sparse Veg Δ (ha/yr)':   f'{sr:+,.1f}',
            'Bare Soil Δ (ha/yr)':     f'{br:+,.1f}',
        })

    df_rate = pd.DataFrame(rows_rate)
    print('Period-by-Period Change Rates:')
    print(df_rate.to_string(index=False))

    # Grouped bar: deforestation vs mine growth ──────────────────────────────
    x = np.arange(len(yr_labels)); w = 0.35
    fig_cr, ax_cr = plt.subplots(figsize=(12, 5))
    b1_bars = ax_cr.bar(x - w/2, defor_rate, w,
                        label='Forest Loss (ha/yr)', color='#c0392b', edgecolor='white')
    b2_bars = ax_cr.bar(x + w/2, mine_rate,  w,
                        label='Mine Growth (ha/yr)', color='#0050C8', edgecolor='white')
    all_rates = [abs(r) for r in defor_rate + mine_rate]
    bump = max(all_rates) * 0.02 if all_rates else 1
    for bar in b1_bars:
        v = bar.get_height()
        ax_cr.text(bar.get_x() + bar.get_width()/2, v + bump,
                   f'{v:+,.0f}', ha='center', fontsize=8, color='#c0392b', fontweight='bold')
    for bar in b2_bars:
        v = bar.get_height()
        ax_cr.text(bar.get_x() + bar.get_width()/2, v + bump,
                   f'{v:+,.0f}', ha='center', fontsize=8, color='#0050C8', fontweight='bold')
    ax_cr.set_xticks(x); ax_cr.set_xticklabels(yr_labels, fontsize=9)
    ax_cr.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax_cr.set_ylabel('ha/yr'); ax_cr.set_title('Annual Change Rate by Period', fontweight='bold')
    ax_cr.legend(fontsize=9)
    ax_cr.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:+,.0f}'))
    ax_cr.spines['top'].set_visible(False); ax_cr.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/07a_rate_defor_vs_mine.png', dpi=150, bbox_inches='tight')
    plt.show(); plt.close()

    # All-4-classes net change bar ───────────────────────────────────────────
    fig_all, ax_all = plt.subplots(figsize=(12, 5))
    x4 = np.arange(len(yr_labels)); w4 = 0.2
    for ki, (rates, cls_n) in enumerate([
        ([-r for r in defor_rate], 'Forest'),
        (sparse_rate,              'Sparse Vegetation'),
        (bare_rate,                'Bare Soil'),
        (mine_rate,                'Mine'),
    ]):
        ax_all.bar(x4 + (ki - 1.5) * w4, rates, w4,
                   label=cls_n,
                   color=np.array(CLASS_COLORS_4[cls_n])/255,
                   edgecolor='white')
    ax_all.set_xticks(x4); ax_all.set_xticklabels(yr_labels, fontsize=9)
    ax_all.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax_all.set_ylabel('Net Change (ha/yr)')
    ax_all.set_title('Net Annual Change — All Classes', fontweight='bold')
    ax_all.legend(fontsize=9)
    ax_all.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:+,.0f}'))
    ax_all.spines['top'].set_visible(False); ax_all.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/07b_rate_all_classes.png', dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
else:
    print('Need at least 2 years to compute change rates.')


In [ ]:
# ── Detailed Per-Year Stats ───────────────────────────────────────────────────
# Inspect each year: horizontal bar with ha + % + Δ vs baseline label,
# plus pie chart comparison vs baseline year.

for sel_yr in years_sorted:
    a_d    = yearly_results[sel_yr]['areas']
    a_base = yearly_results[base]['areas']

    print(f'\n── {sel_yr} ──────────────────────────────────────────')
    for cls_n in CLASSES_4:
        chg = pct_chg(a_d[cls_n], a_base[cls_n]) if a_base[cls_n] > 0 else None
        chg_str = f'  ({chg:+.1f}% vs {base})' if chg is not None else ''
        print(f'  {cls_n:22s}: {a_d[cls_n]:>10,.0f} ha  '
              f'{pct_of_total(a_d[cls_n], sel_yr):5.1f}% of study area{chg_str}')

    # Horizontal bar ──────────────────────────────────────────────────────────
    fig_d, ax_d = plt.subplots(figsize=(11, 4))
    bars_d = ax_d.barh(
        CLASSES_4,
        [a_d[c] for c in CLASSES_4],
        color=[np.array(CLASS_COLORS_4[c])/255 for c in CLASSES_4],
        edgecolor='white', height=0.5)
    max_ha = max(a_d[c] for c in CLASSES_4)
    for bar_d, cls_n in zip(bars_d, CLASSES_4):
        v     = bar_d.get_width()
        pct_v = pct_of_total(v, sel_yr)
        chg_v = pct_chg(v, a_base[cls_n]) if a_base[cls_n] > 0 else None
        chg_str = f'  ({chg_v:+.1f}% vs {base})' if chg_v is not None else ''
        ax_d.text(v + max_ha * 0.01,
                  bar_d.get_y() + bar_d.get_height() / 2,
                  f'{v:,.0f} ha  |  {pct_v:.1f}%{chg_str}',
                  va='center', fontsize=9, fontweight='bold')
    ax_d.set_xlabel('Area (hectares)')
    ax_d.set_title(f'Land Cover — {sel_yr}', fontweight='bold')
    ax_d.set_xlim(0, max_ha * 1.6)
    ax_d.spines['top'].set_visible(False); ax_d.spines['right'].set_visible(False)
    ax_d.spines['left'].set_visible(False); ax_d.tick_params(axis='y', length=0)
    ax_d.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/08_detail_{sel_yr}_bar.png', dpi=150, bbox_inches='tight')
    plt.show(); plt.close()

    # Pie comparison vs baseline (skip if this IS the baseline) ───────────────
    if sel_yr != base:
        pie_cols = [np.array(CLASS_COLORS_4[c])/255 for c in CLASSES_4]
        fig_pie, (ax_p1, ax_p2) = plt.subplots(1, 2, figsize=(12, 5))
        for ax_p, yr_p in [(ax_p1, base), (ax_p2, sel_yr)]:
            vals_p = [yearly_results[yr_p]['areas'][c] for c in CLASSES_4]
            wedges, texts, autotexts = ax_p.pie(
                vals_p, labels=CLASSES_4, colors=pie_cols,
                autopct='%1.1f%%', startangle=90,
                wedgeprops={'edgecolor': 'white', 'linewidth': 1.2})
            for at in autotexts:
                at.set_fontsize(9)
            ax_p.set_title(str(yr_p), fontweight='bold', fontsize=12)
        fig_pie.suptitle('Land Cover Composition Comparison', fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}/08_detail_{sel_yr}_pie.png', dpi=150, bbox_inches='tight')
        plt.show(); plt.close()


---
## Summary of Outputs

| File | Contents |
|------|----------|
| `outputs/01_rgb_preview.png` | True-colour composite after reprojection |
| `outputs/02_spectral_indices.png` | 6-panel index visualisation |
| `outputs/03_kmeans_classification.png` | K-Means result + true colour |
| `outputs/03b_kmeans_pca_scatter.png` | PCA cluster scatter plot |
| `outputs/04_4class_mine_rf.png` | 4-Class Mine RF map |
| `outputs/04b_4class_mine_stats.png` | Area breakdown + feature importance |
| `outputs/04c_confusion_matrix.png` | 4-class confusion matrix |
| `outputs/05_map_<year>.png` | True colour vs classification per year |
| `outputs/05b_first_vs_last.png` | First vs last year 2×2 comparison |
| `outputs/06a_trend_stacked_area.png` | Stacked area chart over time |
| `outputs/06b_trend_pct_composition.png` | % composition stacked bar per year |
| `outputs/06c_trend_forest_vs_mine.png` | Forest vs mine dual-axis annotated |
| `outputs/06d_trend_cumulative_pct.png` | Cumulative % change — all 4 classes |
| `outputs/07a_rate_defor_vs_mine.png` | Annual rate: deforestation vs mine growth |
| `outputs/07b_rate_all_classes.png` | Net annual change — all classes |
| `outputs/08_detail_<year>_bar.png` | Per-year horizontal bar with Δ labels |
| `outputs/08_detail_<year>_pie.png` | Pie comparison vs baseline (non-baseline years) |
| `outputs/multi_year_summary.csv` | Full numeric area table (all years) |
| `outputs/kmeans_model.pkl` | Saved K-Means model |
| `outputs/rf_4class_mine_model.pkl` | Saved 4-class Mine RF |
| `outputs/feature_scaler.pkl` | Saved StandardScaler |
| `outputs/selected_features.txt` | Feature list for inference |
